In [1]:
import polars as pl

In [2]:
df = pl.read_csv("../data/raw/social_media_vs_productivity.csv")

## Impelmentacion basica, lo que quiero es que se ejecuten en paralelo automaticamen 

In [10]:
def etl_pipeline():
    df = pl.read_csv("../data/raw/social_media_vs_productivity.csv")
 

    # Identificar tipos de columnas
    numeric_cols = [col for col in df.columns if df[col].dtype in [pl.Float64, pl.Int64]]
    categorical_cols = [col for col in df.columns if col not in numeric_cols]

    # Transformación con manejo adecuado por tipo de columna
    df_transformed = df.with_columns([
        # Para columnas numéricas: usar mediana
        *[pl.col(col).fill_null(pl.col(col).median()) for col in numeric_cols],

        # Para columnas categóricas: usar modo (valor más frecuente) o un valor predeterminado
        *[pl.col(col).fill_null(pl.col(col).mode().first()) for col in categorical_cols],

        # Características derivadas (solo para columnas numéricas)
        (pl.col("number_of_notifications") * pl.col("daily_social_media_time")).alias("distraction_index"),

        # Categorización
        pl.when(pl.col("daily_social_media_time") > 5)
          .then(pl.lit("high"))
          .when(pl.col("daily_social_media_time") > 2)
          .then(pl.lit("medium"))
          .otherwise(pl.lit("low"))
          .alias("social_usage_category")
    ])

    # Agregaciones
    summary = df_transformed.group_by(["job_type", "social_usage_category"]).agg([
        pl.count().alias("count"),
        pl.mean("actual_productivity_score").alias("avg_productivity"),
        pl.mean("distraction_index").alias("avg_distraction"),
        pl.mean("stress_level").alias("avg_stress")
    ])

    # Retornar resultados
    return df_transformed, summary

# Ejecutar pipeline
transformed_data, summary_data = etl_pipeline()

C:\Users\anoni\AppData\Local\Temp\ipykernel_19980\3459913838.py:31: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("count"),
